<a href="https://colab.research.google.com/github/akshara2006-psit/Flyrank-internship/blob/codespace-miniature-waddle-695v976x9q7jh5rvp/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 modeling lane using the same anonymized content-performance dataset.

**Audit goal:** check whether the Week-5 Random Forest result remains similar when validation is grouped by client, check for feature leakage, inspect real prediction errors, and rewrite claims using evidence-safe language.

The target used here is `decline = 1` when the dataset's `trend_direction` is `down`. This is a retrospective audit label; it should not be treated as a future outcome unless the prediction point and time window are defined separately.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank research report says its growth-prediction model was about **90% accurate on unseen pages from the same brands** and about **75% on brands it had never seen before**.

**My methodology question:** Where exactly does the growth/decline label come from, and does the validation split prevent information from the same brand/client from making the test easier? The same-brand and unseen-brand results are useful comparisons, but I would want the label window and split construction stated clearly before treating the accuracy as evidence of performance on a new client.

### Finding 2 — Refreshing pages

The report states that **7 of 9 strata showed statistically significant refresh lift** in its held-out analysis.

**My methodology question:** How was "refreshed" versus "stale" assigned, and how comparable were the two groups before the refresh? A held-out comparison is helpful, but the claim should distinguish an observed association from a causal effect unless the design controls for important differences between pages that were refreshed and pages that were not.

These are constructive questions about label provenance and validation design, not a judgment that the findings are wrong.

In [1]:
# Imports and data loading
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Works when the notebook is committed under work/notebooks/
candidate_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv",
]

data_path = next((p for p in candidate_paths if __import__("os").path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "Place content_refresh_anonymized.csv in data/raw/ before running the notebook."
    )

df = pd.read_csv(data_path)

# Retrospective audit label from the supplied dataset.
df["decline"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Decline rate:", round(df["decline"].mean(), 4))
df.head(3)

FileNotFoundError: Place content_refresh_anonymized.csv in data/raw/ before running the notebook.

## 2. My model under an honest split (before/after)

For the **before** result, I reproduce a standard random 80/20 split.

For the **after** result, I use an 80/20 split grouped by `client_id`, so a client is not represented in both train and test. This better tests whether the model generalizes beyond the specific clients used for training.

I keep the same Random Forest family and the same target. The main comparison is **Precision@50**, with ROC-AUC and accuracy shown as supporting metrics.

Important: `client_id` is used only for grouping, never as a model feature.

In [ ]:
# Week-5-style feature set:
# remove the target and pseudonymous identifiers, but otherwise retain the available columns.
# This is deliberately audited below for leakage risk.

week5_drop = ["decline", "trend_direction", "trend_pct", "content_id", "client_id"]
X_week5 = df.drop(columns=week5_drop)
y = df["decline"]

categorical = X_week5.select_dtypes(include="object").columns.tolist()
numeric = X_week5.select_dtypes(exclude="object").columns.tolist()

def make_rf_pipeline():
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
        min_samples_leaf=2
    )
    return Pipeline([("prep", prep), ("model", model)])

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    idx = np.argsort(np.asarray(scores))[::-1][:k]
    return float(y_true[idx].mean())

# BEFORE: random split
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

before_model = make_rf_pipeline()
before_model.fit(X_week5.iloc[train_idx], y.iloc[train_idx])
before_scores = before_model.predict_proba(X_week5.iloc[test_idx])[:, 1]
before_pred = (before_scores >= 0.50).astype(int)

# AFTER: grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
group_train_idx, group_test_idx = next(
    gss.split(X_week5, y, groups=df["client_id"])
)

after_model = make_rf_pipeline()
after_model.fit(X_week5.iloc[group_train_idx], y.iloc[group_train_idx])
after_scores = after_model.predict_proba(X_week5.iloc[group_test_idx])[:, 1]
after_pred = (after_scores >= 0.50).astype(int)

comparison = pd.DataFrame([
    {
        "validation": "Random 80/20 (before)",
        "clients_in_test": df.iloc[test_idx]["client_id"].nunique(),
        "Precision@50": precision_at_k(y.iloc[test_idx], before_scores),
        "ROC-AUC": roc_auc_score(y.iloc[test_idx], before_scores),
        "Accuracy": accuracy_score(y.iloc[test_idx], before_pred)
    },
    {
        "validation": "Client-grouped 80/20 (after)",
        "clients_in_test": df.iloc[group_test_idx]["client_id"].nunique(),
        "Precision@50": precision_at_k(y.iloc[group_test_idx], after_scores),
        "ROC-AUC": roc_auc_score(y.iloc[group_test_idx], after_scores),
        "Accuracy": accuracy_score(y.iloc[group_test_idx], after_pred)
    }
])

comparison.round(3)

In [ ]:
# Check whether the client groups are actually disjoint.
train_clients = set(df.iloc[group_train_idx]["client_id"])
test_clients = set(df.iloc[group_test_idx]["client_id"])

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))

assert len(train_clients.intersection(test_clients)) == 0

### Interpretation

The grouped split is a stricter generalization check because test rows come from clients that are absent from training. If the measured score decreases, that is evidence that the random split may have been optimistic for cross-client generalization.

The numbers above are **measured results on this dataset and split**, not a guarantee of performance on future clients.

## 3. Leakage audit

The target `decline` is derived from `trend_direction`. Several fields in the dataset are closely connected to the same measurement window.

I therefore treat the following as leakage-risk features for a forward-looking decline prediction:

- `trend_direction` and `trend_pct`: directly encode the target/trend.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: part of the recent outcome window.
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`: used in the comparison behind the trend.
- `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: derived performance outcomes rather than pre-outcome content attributes.
- `impression_tier`, `position_tier`: derived from performance measures and therefore need careful timing checks.
- `content_id`, `client_id`: identifiers; `client_id` is valid for grouping but not as a model feature.

This audit does not claim every field is definitely unusable in every deployment. The decision depends on the exact prediction point and what information is available at that time.

In [ ]:
# Explicit leakage-risk audit table
audit_rows = [
    ("content_id", "Identifier", "Do not use", "Pseudonymous row identifier"),
    ("client_id", "Grouping identifier", "Do not use as feature", "Use only to create client-grouped validation"),
    ("trend_direction", "Target-derived", "Exclude", "Directly defines the decline label"),
    ("trend_pct", "Target-derived", "Exclude", "Encodes the same trend comparison"),
    ("impressions_last_30d", "Outcome-window metric", "Exclude for forward prediction", "Uses recent performance"),
    ("clicks_last_30d", "Outcome-window metric", "Exclude for forward prediction", "Uses recent performance"),
    ("sessions_last_30d", "Outcome-window metric", "Exclude for forward prediction", "Uses recent performance"),
    ("impressions_prev_30d", "Outcome-window metric", "Exclude for forward prediction", "Part of trend comparison"),
    ("clicks_prev_30d", "Outcome-window metric", "Exclude for forward prediction", "Part of trend comparison"),
    ("sessions_prev_30d", "Outcome-window metric", "Exclude for forward prediction", "Part of trend comparison"),
    ("ctr", "Derived performance", "Exclude for forward prediction", "Computed from clicks/impressions"),
    ("engagement_rate", "Derived performance", "Exclude for forward prediction", "Computed from engagement outcomes"),
    ("scroll_rate", "Derived performance", "Exclude for forward prediction", "Computed from scroll outcomes"),
    ("ai_traffic_pct", "Derived performance", "Review timing", "Uses observed traffic mix"),
    ("impression_tier", "Derived feature", "Review timing", "Derived from impression performance"),
    ("position_tier", "Derived feature", "Review timing", "Derived from position performance"),
]

leakage_audit = pd.DataFrame(
    audit_rows,
    columns=["feature", "risk_type", "decision", "reason"]
)

leakage_audit

In [ ]:
# Build a more conservative feature set for a forward-looking sensitivity check.
safe_drop = [
    "decline", "trend_direction", "trend_pct",
    "content_id", "client_id",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier"
]

X_safe = df.drop(columns=safe_drop)

safe_cat = X_safe.select_dtypes(include="object").columns.tolist()
safe_num = X_safe.select_dtypes(exclude="object").columns.tolist()

def make_safe_rf():
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), safe_num),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), safe_cat)
    ])
    return Pipeline([
        ("prep", prep),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
            min_samples_leaf=2
        ))
    ])

safe_model = make_safe_rf()
safe_model.fit(X_safe.iloc[group_train_idx], y.iloc[group_train_idx])
safe_scores = safe_model.predict_proba(X_safe.iloc[group_test_idx])[:, 1]
safe_pred = (safe_scores >= 0.50).astype(int)

safe_result = pd.DataFrame([{
    "validation": "Client-grouped + leakage-reduced features",
    "Precision@50": precision_at_k(y.iloc[group_test_idx], safe_scores),
    "ROC-AUC": roc_auc_score(y.iloc[group_test_idx], safe_scores),
    "Accuracy": accuracy_score(y.iloc[group_test_idx], safe_pred)
}])

safe_result.round(3)

### What this leakage check changes

The conservative sensitivity check is intentionally stricter. If its measured performance is lower than the Week-5-style feature set, I should not present the higher score as the strongest evidence for a forward-looking system.

The safe interpretation is that the original feature set contains variables whose timing must be resolved before the model can be described as a future prediction system.

## 4. Claim rewrite

### Original bold claim

> "The Random Forest predicts which content will decline accurately and can be used to identify future declining pages."

### Evidence-safe rewrite

> "On this 30,000-row anonymized dataset, the Random Forest showed measured performance for identifying rows labeled as declining. Performance was higher under the random split than under the client-grouped split, and several performance-derived features were identified as leakage risks for a forward-looking prediction task. The current result is therefore best described as **directional decision-support**, not proof of future-page performance."

In [ ]:
# Real error examples from the conservative grouped evaluation.
# IDs are intentionally removed from the displayed output.

error_view = df.iloc[group_test_idx].copy()
error_view["predicted_decline_probability"] = safe_scores
error_view["predicted_decline"] = safe_pred
error_view["actual_decline"] = y.iloc[group_test_idx].to_numpy()

false_positives = error_view[
    (error_view["actual_decline"] == 0) &
    (error_view["predicted_decline"] == 1)
].copy()

false_negatives = error_view[
    (error_view["actual_decline"] == 1) &
    (error_view["predicted_decline"] == 0)
].copy()

# Show only non-identifying, model-relevant fields.
display_cols = [
    "search_volume", "competition", "competition_level",
    "content_type", "main_intent", "word_count",
    "content_age_days", "days_since_last_update",
    "predicted_decline_probability", "actual_decline"
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives")
display(false_positives[display_cols].head(5).round(3))

print("\nExample false negatives")
display(false_negatives[display_cols].head(5).round(3))

In [ ]:
# Simple aggregate error view: useful for interpretation without exposing row identifiers.
error_summary = pd.DataFrame({
    "group": ["False positives", "False negatives"],
    "count": [len(false_positives), len(false_negatives)]
})
error_summary

### Error interpretation

The error examples are more useful than treating one metric as the whole story. False positives are rows the model treated as likely decline but whose label was not decline; false negatives are declining rows the model did not flag.

These examples show where the model's decision-support signal is imperfect. They should be used to guide feature review and future validation, not as evidence that a particular content change would cause improvement.

## Self-check

- [x] Two paper findings are named and each has a constructive methodology question.
- [x] Before/after validation uses the same Random Forest family with a random split versus a client-grouped split.
- [x] `client_id` is used for grouping only, never as a model feature.
- [x] A leakage audit identifies target-derived and outcome-window features.
- [x] A conservative leakage-reduced sensitivity check is included.
- [x] Real false-positive and false-negative examples are shown without client IDs.
- [x] Claims use careful language such as **observed, measured, directional, and decision-support**.
- [x] No client names, URLs, private queries, or raw content identifiers are displayed.
- [ ] Run the notebook top-to-bottom in your repo and commit the executed version under `work/notebooks/w06_validation_audit.ipynb`.